# 8.2 Institutional Pattern Discovery II: Unsupervised Learning with Survey Data

## Introduction



In the previous sections, we used **supervised** models to predict known outcomes like retention or stress. However, some of the most valuable institutional insights come from **unsupervised learning**, where we don't have a specific "label" to predict.

Instead, we ask the data to reveal the natural, hidden groupings within our student population. This allows us to move beyond broad demographic categories and discover **Student Personas** based on their actual behaviors and voices.

<br>

#### **Guiding Questions**
1. Can we discover natural groupings of students by combining structured academic records with unstructured survey text?
2. How do we interpret and translate these mathematical "clusters" into actionable institutional strategies?

<br>

#### **Learning Objectives**
By the end of this notebook, you will be able to:
* **Preprocess** a mixed dataset (numeric, categorical, and text) into a unified, model-ready matrix.
* **Apply K-Means Clustering** to find latent student segments.
* **Determine the Optimal K** using the Elbow Method (Inertia).
* **Profile and Interpret** clusters to communicate high-impact findings to campus stakeholders.


## 1. Why Unsupervised Learning in IR?


Supervised learning tells us *who* is likely to leave or succeed. Unsupervised learning tells us **who our students actually are**.

By using **K-Means Clustering**, we can surface distinct segments that might otherwise remain hidden in large datasets, such as:
* **The High-Engagement Thrivers:** Students with high GPAs and positive survey sentiment who may be perfect candidates for peer-mentorship roles.
* **The "Silent" At-Risk Group:** Students with moderate grades but low survey engagement: a subtle early-warning signal that administrative data often misses.
* **The Transition-Strugglers:** First-gen or transfer students whose open-ended comments reflect high academic anxiety despite stable mid-term grades.

> **Institutional Perspective:** Clustering turns "Big Data" into "Human Personas." Instead of looking at 5,000 individual rows, we can look at 3 or 4 meaningful student profiles, allowing for much more personalized and effective resource allocation.

## 2. Setup and Data Preparation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

pd.options.display.max_columns = None
np.random.seed(42)
random.seed(42)

filepath = '/content/drive/MyDrive/IR ML Cert/MLCert Course 3/Course 3 Data/'
df_ml_train = pd.read_csv(f'{filepath}ML_SURVEY_MASTER_TRAIN.csv')

df_ml_train

,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,GENDER_Female,GENDER_Male,RACE_ETHNICITY_Asian,RACE_ETHNICITY_Black or African American,RACE_ETHNICITY_Hispanic,RACE_ETHNICITY_Nonresident alien,RACE_ETHNICITY_Other,RACE_ETHNICITY_Two or More Races,RACE_ETHNICITY_White,FIRST_GEN_STATUS_Continuing Generation,FIRST_GEN_STATUS_First Generation,FIRST_GEN_STATUS_Unknown,TEXT_PC1,TEXT_PC2,TEXT_PC3,TEXT_PC4,TEXT_PC5,TEXT_PC6,TEXT_PC7,TEXT_PC8,TEXT_PC9,TEXT_PC10,TEXT_PC11,TEXT_PC12,TEXT_PC13,TEXT_PC14,TEXT_PC15,TEXT_PC16,TEXT_PC17,TEXT_PC18,TEXT_PC19,TEXT_PC20,TEXT_PC21,TEXT_PC22,TEXT_PC23,TEXT_PC24,TEXT_PC25,TEXT_PC26,TEXT_PC27,TEXT_PC28,TEXT_PC29,TEXT_PC30,TEXT_PC31,TEXT_PC32,TEXT_PC33,SEM_3_STATUS
0,0.733231,1.000000,0.946429,0.0000,0.000000,0.776601,0.298144,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.205242,-0.233176,-0.131520,0.102125,0.134499,0.460959,0.172846,-0.333283,-0.411112,-0.412847,-0.071479,0.074861,-0.050296,-0.144858,0.034223,-0.002189,0.004773,-0.183842,0.001556,-0.004276,-0.020279,0.006026,0.012382,-0.041590,0.005752,-0.032502,0.024440,0.001746,0.022948,-0.029788,-0.014734,0.003449,0.020035,E
1,0.528527,0.750000,0.625000,0.0000,0.000000,-0.390977,-0.453475,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.083616,-0.075247,-0.057068,-0.078357,0.003775,0.074458,-0.113168,-0.120853,0.109394,0.167026,0.049050,0.022447,0.100718,0.025883,0.089616,0.036634,0.065520,0.189085,-0.197031,0.245888,-0.007957,-0.020323,-0.005028,-0.240896,0.639755,0.172229,0.071260,0.070891,0.031551,-0.062611,-0.178118,-0.082006,0.081144,E
2,0.696608,0.950000,0.900000,0.0000,0.000000,0.776601,0.673954,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.008444,0.074317,0.001697,0.131197,-0.155332,0.575747,-0.268791,0.676373,0.162704,-0.046143,-0.034399,0.068832,0.015647,-0.053980,-0.010688,0.028420,0.008349,-0.092916,-0.073729,0.004037,-0.027295,0.051875,0.001716,-0.031478,-0.027612,-0.017511,0.024934,-0.017812,0.008592,-0.012415,0.005109,-0.027956,-0.027070,E
3,0.689283,0.390625,0.250000,0.5625,0.666667,1.165793,-1.580904,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.023884,-0.107904,-0.023733,0.550062,-0.339983,-0.307736,-0.002587,-0.009734,0.047827,-0.026779,0.424551,0.123250,0.062017,-0.449034,0.046378,0.032350,0.055462,-0.027938,0.004890,-0.006380,0.004925,-0.006452,-0.000709,0.047198,0.031964,-0.035746,0.022827,-0.032426,0.002429,0.018500,-0.019824,0.016228,0.008200,E
4,0.662298,0.884615,0.942308,0.0000,0.000000,-0.001785,-0.077666,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,-0.484680,-0.123860,-0.200994,-0.183085,0.142164,-0.116969,0.251671,0.036934,0.459540,0.029444,-0.263991,0.077692,-0.126198,-0.223807,-0.191222,0.126753,0.022958,-0.010518,0.019908,-0.038438,-0.015518,0.046624,-0.000035,0.046158,-0.024800,0.006739,0.029115,-0.084122,-0.065985,-0.105432,0.009201,0.027391,-0.048596,E
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19839,0.635698,0.750000,0.846154,0.0000,0.000000,-1.169363,-0.077666,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.029939,-0.430074,0.046096,-0.072726,0.188192,0.203552,-0.301012,-0.306915,0.194181,0.429833,0.131604,0.040831,0.375180,0.097139,0.088086,0.035010,0.012819,-0.127662,-0.022031,-0.036158,-0.067736,0.048636,0.026807,0.032716,-0.074652,-0.042389,0.011844,-0.003245,-0.030938,0.089311,-0.001449,-0.016028,0.011645,E
19840,0.824210,0.800000,1.000000,0.0000,0.000000,0.776601,0.673954,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,-0.484680,-0.123860,-0.200994,-0.183085,0.142164,-0.116969,0.251671,0.036934,0.459540,0.029444,-0.263991,0.077692,-0.126198,-0.223807,-0.191222,0.126753,0.022958,-0.010518,0.019908,-0.038438,-0.015518,0.046624,-0.000035,0.046158,-0.024800,0.006739,0.029115,-0.084122,-0.065985,-0.105432,0.009201,0.027391,-0.048596,E
19841,0.807247,0.850000,0.859375,0.0000,0.000000,0.7766


## 3. Preprocessing for Cluster Analysis


As established earlier in the certificate program, we use the `ColumnTransformer` to handle mixed data types. However, for **K-Means Clustering**, this step is even more critical because the algorithm is entirely "distance-based."

If we do not scale, features with larger raw numbers (like Units Attempted) will mathematically "overwhelm" more nuanced metrics (like GPA), leading to clusters that only reflect credit loads rather than holistic student profiles.

| Feature Category | Scaling Strategy | IR Strategy for Clustering |
| :--- | :--- | :--- |
| **GPA & DFW Rates** | **MinMaxScaler** | Keeps academic performance within a comparable 0.0–1.0 range. |
| **Credit Volume** | **StandardScaler** | Normalizes "Units Attempted" so they don't dominate the distance calculation. |
| **Demographics** | **OneHotEncoder** | Translates categorical identity into numerical coordinates for the model. |

<br>

> **Instructor Perspective:** In Course 2, you learned the *code* for this pipeline. In Course 3, focus on the **Impact**. Scaling ensures that a 0.5-point difference in GPA is treated with the same mathematical "weight" as a 3-unit difference in course load.


In [42]:
X_train = df_ml_train.drop(columns = ['SEM_3_STATUS'])

To be able to identify characteristics o students in each cluster, it is helpful to have a version of th DataFrame before the demographic and academic variables are preprocessed.

In [23]:
df_training = pd.read_csv(f'{filepath}training.csv')
df_training1 = df_training.drop(columns = ['SEM_3_STATUS'])
X_raw = pd.concat([df_training1,X_train.iloc[:,19:]],axis=1)
X_raw

,SID,COHORT,RACE_ETHNICITY,GENDER,FIRST_GEN_STATUS,HS_GPA,HS_MATH_GPA,HS_ENGL_GPA,COLLEGE,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2,UNITS_ATTEMPTED_3,UNITS_COMPLETED_1,UNITS_COMPLETED_2,UNITS_COMPLETED_3,DFW_UNITS_1,DFW_UNITS_2,DFW_UNITS_3,GPA_1,GPA_2,GPA_3,DFW_RATE_1,DFW_RATE_2,GRADE_POINTS_1,GRADE_POINTS_2,GRADE_POINTS_3,TEXT_PC1,TEXT_PC2,TEXT_PC3,TEXT_PC4,TEXT_PC5,TEXT_PC6,TEXT_PC7,TEXT_PC8,TEXT_PC9,TEXT_PC10,TEXT_PC11,TEXT_PC12,TEXT_PC13,TEXT_PC14,TEXT_PC15,TEXT_PC16,TEXT_PC17,TEXT_PC18,TEXT_PC19,TEXT_PC20,TEXT_PC21,TEXT_PC22,TEXT_PC23,TEXT_PC24,TEXT_PC25,TEXT_PC26,TEXT_PC27,TEXT_PC28,TEXT_PC29,TEXT_PC30,TEXT_PC31,TEXT_PC32,TEXT_PC33
0,UHDOP5522,Fall 2020,Asian,Female,Continuing Generation,3.720,3.2,3.400,Visual & Performing Arts,15.0,14.0,15.0,15.0,15.0,16.0,0.0,0.0,0.0,4.000000,3.785714,4.000000,0.0000,0.000000,60.0,53.0,60.0,-0.205242,-0.233176,-0.131520,0.102125,0.134499,0.460959,0.172846,-0.333283,-0.411112,-0.412847,-0.071479,0.074861,-0.050296,-0.144858,0.034223,-0.002189,0.004773,-0.183842,0.001556,-0.004276,-0.020279,0.006026,0.012382,-0.041590,0.005752,-0.032502,0.024440,0.001746,0.022948,-0.029788,-0.014734,0.003449,0.020035
1,UHE842CU6,Fall 2021,Black or African American,Female,Continuing Generation,3.189,2.6,3.750,Visual & Performing Arts,12.0,12.0,12.0,12.0,12.0,6.0,3.0,4.0,12.0,3.000000,2.500000,1.500000,0.0000,0.000000,36.0,30.0,18.0,-0.083616,-0.075247,-0.057068,-0.078357,0.003775,0.074458,-0.113168,-0.120853,0.109394,0.167026,0.049050,0.022447,0.100718,0.025883,0.089616,0.036634,0.065520,0.189085,-0.197031,0.245888,-0.007957,-0.020323,-0.005028,-0.240896,0.639755,0.172229,0.071260,0.070891,0.031551,-0.062611,-0.178118,-0.082006,0.081144
2,UHJFT1JAB,Fall 2018,Asian,Female,Continuing Generation,3.625,3.4,3.500,Visual & Performing Arts,15.0,15.0,15.0,15.0,16.0,16.0,0.0,0.0,0.0,3.800000,3.600000,3.600000,0.0000,0.000000,57.0,54.0,54.0,0.008444,0.074317,0.001697,0.131197,-0.155332,0.575747,-0.268791,0.676373,0.162704,-0.046143,-0.034399,0.068832,0.015647,-0.053980,-0.010688,0.028420,0.008349,-0.092916,-0.073729,0.004037,-0.027295,0.051875,0.001716,-0.031478,-0.027612,-0.017511,0.024934,-0.017812,0.008592,-0.012415,0.005109,-0.027956,-0.027070
3,UHKF05TAF,Fall 2018,Hispanic,Female,First Generation,3.606,3.0,3.375,Letters & Humanities,16.0,9.0,12.0,7.0,3.0,12.0,9.0,9.0,3.0,1.562500,1.000000,2.500000,0.5625,0.666667,25.0,9.0,30.0,-0.023884,-0.107904,-0.023733,0.550062,-0.339983,-0.307736,-0.002587,-0.009734,0.047827,-0.026779,0.424551,0.123250,0.062017,-0.449034,0.046378,0.032350,0.055462,-0.027938,0.004890,-0.006380,0.004925,-0.006452,-0.000709,0.047198,0.031964,-0.035746,0.022827,-0.032426,0.002429,0.018500,-0.019824,0.016228,0.008200
4,UHKKQ8UY5,Fall 2021,Hispanic,Male,Continuing Generation,3.536,2.5,2.625,Letters & Humanities,13.0,13.0,15.0,13.0,13.0,15.0,0.0,0.0,0.0,3.538462,3.769231,3.400000,0.0000,0.000000,46.0,49.0,51.0,-0.484680,-0.123860,-0.200994,-0.183085,0.142164,-0.116969,0.251671,0.036934,0.459540,0.029444,-0.263991,0.077692,-0.126198,-0.223807,-0.191222,0.126753,0.022958,-0.010518,0.019908,-0.038438,-0.015518,0.046624,-0.000035,0.046158,-0.024800,0.006739,0.029115,-0.084122,-0.065985,-0.105432,0.009201,0.027391,-0.048596
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19839,N2HOWBXJM,Fall 2019,Hispanic,Female,First Generation,3.467,3.4,3.833,Natural and Mathematical Sciences,10.0,13.0,14.0,10.0,13.0,14.0,0.0,0.0,0.0,3.000000,3.384615,2.428571,0.0000,0.000000,30.0,44.0,34.0,-0.029939,-0.430074,0.046096,-0.072726,0.188192,0.203552,-0.301012,-0.306915,0.194181,0.429833,0.131604,0.040831,0.375180,0.097139,0.088086,0.035010,0.012819,-0.127662,-0.022031,-0.036158,-0.067736,0.048636,0.026807,0.032716,-0.074652,-0.042389,0.011844,-0.003245,-0.030938,0.089311,-0.001449,-0.016028,0.011645
19840,N2JGRD0V6,Fall 2018,White,Fema


## 4. Reduce TF-IDF Text Features with PCA


Free-response text vectorized with TF-IDF produces hundreds of columns, far too many to cluster directly alongside our ~19 structured features. To prevent the text data from mathematically "overwhelming" our academic metrics, we use **PCA (Principal Component Analysis)**.

PCA compresses the high-dimensional word matrix into a small number of "Principal Components" that capture the core variance (the "essence") of the student voice.

**Determining the Optimal Number of Components:**


We use an **elbow plot** (Cumulative Explained Variance) to choose our cutoff. In institutional research, capturing **80% of the variance** is a common standard, as it retains the primary emotional and topical signals while discarding the "noise" of rare or unique words.

<br>

> **Instructor Perspective:** Think of PCA as an "Equalizer." Without it, the 250+ word columns would act like 250+ separate votes in the clustering algorithm, while your GPA only gets one vote. PCA condenses those 250+ votes into roughly 30, giving the "Student Voice" and "Student Records" a more balanced influence on the final personas.



Once the optimal threshold is found, we project the 250+ raw word columns into 33 Principal Components (`TEXT_PC1`, `TEXT_PC2`, etc.) to create a balanced feature set for clustering.

We use `pd.concat` with `axis=1` to join our structured features and our compressed text components side-by-side.

> **Technical Note:** We cast all column names to strings at this stage. This is a requirement for the K-Means algorithm, which expects a consistent header format to perform distance calculations.


## 6. Choose Number of Clusters: Elbow Method


K-Means requires us to specify $K$ (the number of clusters) upfront. Because there is no "correct" label in unsupervised learning, we use the **Elbow Method** to find a mathematically sound value for $K$.

We plot **Inertia** (the sum of squared distances from each student point to their assigned cluster center). As $K$ increases, inertia naturally drops. We are looking for the "Elbow"—the specific point where adding more clusters no longer yields a significant decrease in inertia.



> **Instructor Perspective:** In Institutional Research, choosing $K$ is a balance between **Granularity** and **Actionability**. A $K$ of 20 might be mathematically precise, but a Provost cannot design 20 different student success initiatives. We typically look for $K$ values between 3 and 6 to ensure the resulting personas are distinct enough to drive real campus policy.


We iterate through a range of potential cluster counts ($K=1$ to $10$), calculating the inertia at each step to visualize the trade-off between model complexity and group cohesion.

In [22]:
inertia = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_train)
    inertia.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertia,
              markers=True,
              labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'},
              title='Elbow Method — Choosing K for K-Means')
fig.show()
print("Look for the point where the curve flattens — that is your elbow.")


Look for the point where the curve flattens — that is your elbow.


## 7. Fit K-Means & Assign Cluster Labels





Now that we have identified the "Elbow" at **K=3**, we fit our final model. This process assigns every student in our Fall 2019 cohort to one of three distinct groups based on the mathematical similarities in their academic records and survey voices.

> **Technical Note:** We use `n_init=10` to ensure the model runs multiple times with different starting points, selecting the best result to avoid "local minima" (sub-optimal groupings).


In [28]:
OPTIMAL_K = 3  # ← update this based on the elbow plot above

kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
kmeans.fit(X_train)

X_train = X_train.copy()
X_raw['Cluster'] = kmeans.labels_

print("Cluster sizes:")
print(X_raw['Cluster'].value_counts().sort_index())


Cluster sizes:
Cluster
0    10614
1     2415
2     6815
Name: count, dtype: int64


## 8. Visualize Clusters with PCA



Our master matrix has **52 dimensions** (19 structured features + 33 text components). Because the human eye cannot see in 52D, we use PCA once more to project the data down to **two coordinates (PC1 and PC2)** for visualization.

**Important Distinction:** * The **Clustering** was done using all 52 features to ensure maximum accuracy.
* The **Visualization** uses only 2 features so we can "see" the separation between our student groups.



> **Key note:** Look for the "overlap" in the scatter plot. In social science data, clusters are rarely perfectly isolated circles; students are complex and often sit on the borders between personas. Our goal is to find the **dominant trend** in each group.



In [29]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(X_train)

df_plot = pd.DataFrame({
    'PC1': coords[:, 0],
    'PC2': coords[:, 1],
    'Cluster': X_raw['Cluster'].astype(str),
    'GPA': X_raw['HS_GPA'].round(2),
    'First_Gen': X_raw['FIRST_GEN_STATUS'],
    'Gender': X_raw['GENDER']
}, index=X_raw.index)

fig = px.scatter(
    df_plot, x='PC1', y='PC2',
    color='Cluster',
    hover_data=['GPA', 'First_Gen', 'Gender'],
    title=f'K-Means Clusters (K={OPTIMAL_K}) — Visualized with 2D PCA',
    labels={'PC1': 'Principal Component 1', 'PC2': 'Principal Component 2'}
)
fig.update_traces(marker=dict(size=6, opacity=0.75))
fig.show()


Here we can see three distinct groupings geometrically. However there is not a natural seperation between the clusters. This may imply students on the borders are virtually indistinguishible in terms of cluster membership.


## 9. Profile and Interpret Clusters


The scatter plot shows *where* clusters sit in 2D space, but to take action, we must understand *who* is in each group. We "profile" these clusters by calculating the average values for our key academic and demographic features.

In Institutional Research, this step is known as **Persona Development**. We are looking for the "Typical Student" within each cluster so we can tailor support services to their specific needs.



> **Instructor Perspective:** Don't just look at the numbers—look for the **Story**. If Cluster 2 has a high DFW rate and mentions "work-life balance" in their comments, they aren't just a "low-performing group"; they are a "High-Obligation" group that may need more flexible course scheduling.


We calculate the mean for each academic metric by cluster to identify the 'Performance Fingerprint' of each student group.

In [30]:
# Numeric profile
profile_cols = ['HS_GPA', 'GPA_1', 'GPA_2', 'DFW_RATE_1', 'DFW_RATE_2',
                'UNITS_ATTEMPTED_1', 'UNITS_ATTEMPTED_2']
profile_cols = [c for c in profile_cols if c in X_raw.columns]

cluster_profile = X_raw.groupby('Cluster')[profile_cols].mean().round(3)
print("── Cluster Numeric Profile ──")
cluster_profile


── Cluster Numeric Profile ──


,HS_GPA,GPA_1,GPA_2,DFW_RATE_1,DFW_RATE_2,UNITS_ATTEMPTED_1,UNITS_ATTEMPTED_2
Cluster,,,,,,,
0,3.696,3.215,3.138,0.074,0.087,14.650,14.311
1,3.507,2.587,2.348,0.205,0.280,12.429,7.925
2,3.591,2.823,2.827,0.151,0.160,10.646,13.359


By normalizing the distributions of Gender and First-Gen status, we can see if certain demographic groups are disproportionately represented in specific success clusters.

In [33]:
# Categorical breakdown
print("── First-Generation Status by Cluster ──")
pd.crosstab(X_raw['Cluster'], X_raw['FIRST_GEN_STATUS'], normalize='index').round(3)


── First-Generation Status by Cluster ──


FIRST_GEN_STATUS,Continuing Generation,First Generation,Unknown
Cluster,,,
0,0.703,0.236,0.061
1,0.609,0.299,0.092
2,0.503,0.382,0.115


In [32]:
print("\n── Gender by Cluster ──")
pd.crosstab(X_raw['Cluster'], X_raw['GENDER'], normalize='index').round(3)


── Gender by Cluster ──


GENDER,Female,Male
Cluster,,
0,0.594,0.406
1,0.552,0.448
2,0.622,0.378


Finally, we pull raw survey text from each cluster to hear the 'Student Voice' behind the numbers, allowing us to validate the mathematical groupings with qualitative context.

In [36]:
# Sample comments from each cluster

ML_Survey_Data = pd.read_csv(f'{filepath}ML_Survey_Data.csv')

print("── Representative Comments by Cluster ──")
for c in sorted(X_raw['Cluster'].unique()):
    index = X_raw[X_raw['Cluster'] == c].index
    subset = ML_Survey_Data.iloc[index,:]
    examples = subset.sample(min(2, len(subset)), random_state=42)['Free_Response_Text'].tolist()
    print(f"\nCluster {c} (n={len(subset)}):")
    for ex in examples:
        print(f"  • {ex}")

── Representative Comments by Cluster ──

Cluster 0 (n=10614):
  • The transition to college has been tougher than I expected, but I'm determined to grow.
  • I do best when I get clear feedback on what to improve, especially on writing and projects.

Cluster 1 (n=2415):
  • Study groups help, but I mostly use them to compare approaches after I’ve done my own work first.
  • I do best when I get clear feedback on what to improve, especially on writing and projects.

Cluster 2 (n=6815):
  • I know support exists, but it can be hard to reach out when I’m already stressed.
  • I am adapting to college life and finding my rhythm, learning new things every day.


This gives us some insight into student perspective by cluster. We can combine this with topic modeling and sentiment analysis for even richer insight. Finally, let's look at aggregate numerical statistics by cluster:

In [41]:
# Visualize numeric profile as a heatmap
import plotly.figure_factory as ff

z = cluster_profile.values.tolist()
x = cluster_profile.columns.tolist()
y = [f'Cluster {i}' for i in cluster_profile.index]

fig = px.imshow(
    cluster_profile,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='Blues',
    title='Cluster Profiles — Average Feature Values',
    labels={'x': 'Feature', 'y': 'Cluster', 'color': 'Mean Value'}
)
fig.show()


## 10. Wrap-Up



**What you did:**
- Preprocessed a mixed dataset (numeric + categorical + free-text) into a single feature matrix
- Used **PCA** to compress TF-IDF text vectors into a manageable number of components
- Applied the **Elbow Method** to choose a reasonable K
- Fit **K-Means** and assigned each student a cluster label
- Profiled clusters by GPA, DFW rates, demographic composition, and survey tone

**How to use clusters in IR practice:**
- Flag clusters with low GPA + high DFW for early-alert advising outreach
- Identify clusters dominated by first-gen students who express struggle — target programming
- Track cluster membership over cohorts to evaluate intervention effectiveness

**Key caution:** Clusters are patterns, not ground truth. Always validate with domain expertise and be cautious about using cluster labels in high-stakes decisions without further analysis.

